In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filename = "tinyshakespeare.txt"

# Download the file
urllib.request.urlretrieve(url, filename)

# Read the file to confirm it worked
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

print("Success! Dataset downloaded.")
print("Total characters:", len(text))

Success! Dataset downloaded.
Total characters: 1115394


In [ ]:
import numpy as np

# Find all the unique characters in the dataset (our "vocabulary")
vocab = sorted(set(text))
print(f"There are {len(vocab)} unique characters in the vocabulary.")

# Create a dictionary to map each character to a number, and vice versa
char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = np.array(vocab)

# Convert the entire text dataset into a massive array of numbers
text_as_int = np.array([char_to_index[c] for c in text])

print("First 15 characters mapped to numbers:")
print(f"{repr(text[:15])} ---> {text_as_int[:15]}")

There are 65 unique characters in the vocabulary.
First 15 characters mapped to numbers:
'First Citizen:\n' ---> [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0]


In [ ]:
import tensorflow as tf

seq_length = 100 # The model will look at 100 characters at a time
examples_per_epoch = len(text) // (seq_length + 1)

# Create training sequences
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

# Function to split the sequence into input and target
def split_input_target(sequence):
    input_text = sequence[:-1]  # Everything except the last character
    target_text = sequence[1:]  # Everything except the first character
    return input_text, target_text

# Apply the function to all data
dataset = sequences.map(split_input_target)

# Shuffle the data and pack it into batches to feed the model efficiently
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE)
)

print("Sequences successfully constructed and batched!")

Sequences successfully constructed and batched!


In [ ]:
# Define identical experimental conditions
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 512

# 1. Simple RNN Model
def build_simplernn():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.SimpleRNN(rnn_units, return_sequences=True),
        tf.keras.layers.Dense(vocab_size)
    ])

# 2. Bidirectional RNN Model
def build_birnn():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        # Divide rnn_units by 2 because it runs in two directions (forward and backward)
        tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(rnn_units // 2, return_sequences=True)),
        tf.keras.layers.Dense(vocab_size)
    ])

# 3. LSTM Model
def build_lstm():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.LSTM(rnn_units, return_sequences=True),
        tf.keras.layers.Dense(vocab_size)
    ])

# 4. GRU Model
def build_gru():
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.GRU(rnn_units, return_sequences=True),
        tf.keras.layers.Dense(vocab_size)
    ])

# Create the actual models in memory
model_rnn = build_simplernn()
model_birnn = build_birnn()
model_lstm = build_lstm()
model_gru = build_gru()

print("All four model architectures successfully built!")

All four model architectures successfully built!


In [ ]:
import time
import math

# Define how we calculate errors (Loss)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Create a function to train a model and record its quantitative measures
def train_and_evaluate(model, name, epochs=5):
    print(f"\n{'='*30}")
    print(f"--- Training {name} ---")
    print(f"{'='*30}")

    # Compile the model with the Adam optimizer and accuracy tracking
    model.compile(optimizer='adam', loss=loss_fn, metrics=['sparse_categorical_accuracy'])

    # Start the timer
    start_time = time.time()

    # Train the model! (We use 5 epochs to keep it relatively fast, but it will still learn)
    history = model.fit(dataset, epochs=epochs)

    # Stop the timer
    end_time = time.time()

    # Calculate final metrics
    training_time = end_time - start_time
    final_loss = history.history['loss'][-1]
    final_accuracy = history.history['sparse_categorical_accuracy'][-1]
    perplexity = math.exp(final_loss)

    print(f"\n[ {name} Final Results ]")
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Final Loss: {final_loss:.4f}")
    print(f"Prediction Accuracy: {final_accuracy:.4f}")
    print(f"Perplexity: {perplexity:.4f}\n")

    return history

# Run the training process for all four models
history_rnn = train_and_evaluate(model_rnn, "Simple RNN")
history_birnn = train_and_evaluate(model_birnn, "Bidirectional RNN")
history_lstm = train_and_evaluate(model_lstm, "LSTM")
history_gru = train_and_evaluate(model_gru, "GRU")


--- Training Simple RNN ---
Epoch 1/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 2.4851 - sparse_categorical_accuracy: 0.3195
Epoch 2/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 1.9784 - sparse_categorical_accuracy: 0.4253
Epoch 3/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 1.8013 - sparse_categorical_accuracy: 0.4706
Epoch 4/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 1.6926 - sparse_categorical_accuracy: 0.4991
Epoch 5/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 1.6194 - sparse_categorical_accuracy: 0.5175

[ Simple RNN Final Results ]
Training Time: 37.26 seconds
Final Loss: 1.6194
Prediction Accuracy: 0.5175
Perplexity: 5.0500


--- Training Bidirectional RNN ---
Epoch 1/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 0.4930 - sparse_categorical_accuracy: 0.9012
Epoch 2/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0300 - sparse_categorical_accuracy: 0.9930
Epoch 3/5
172/172 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 0.0253 - sp

In [ ]:
# Wrap models with a generation class to handle sampling
class TextGenerator:
    def __init__(self, model, char_to_index, index_to_char):
        self.model = model
        self.char_to_index = char_to_index
        self.index_to_char = index_to_char

    def generate_text(self, seed_string, num_generate=200, temperature=1.0):
        # Convert seed string to numerical IDs
        input_chars = [self.char_to_index[c] for c in seed_string]
        input_tensor = tf.expand_dims(input_chars, 0)

        generated_chars = []

        for _ in range(num_generate):
            predictions = self.model(input_tensor)

            # Extract logits for the final character step
            predictions = predictions[:, -1, :]

            # Apply temperature scaling
            predictions = predictions / temperature

            # Sample the next token from the categorical distribution
            predicted_id = tf.random.categorical(predictions, num_samples=1)
            predicted_id = tf.squeeze(predicted_id, axis=-1).numpy()[0]

            # Append predicted character to the sequence
            generated_chars.append(self.index_to_char[predicted_id])

            # Feed predicted character back into the input sequence
            input_tensor = tf.concat([input_tensor, tf.constant([[predicted_id]])], axis=1)

            # Keep context window bounded to prevent memory overflow
            if input_tensor.shape[1] > 100:
                input_tensor = input_tensor[:, -100:]

        return seed_string + "".join(generated_chars)

# Initialize generator wrappers for all four models
gen_rnn = TextGenerator(model_rnn, char_to_index, index_to_char)
gen_birnn = TextGenerator(model_birnn, char_to_index, index_to_char)
gen_lstm = TextGenerator(model_lstm, char_to_index, index_to_char)
gen_gru = TextGenerator(model_gru, char_to_index, index_to_char)

# Experimental settings: same seed sequence across three temperature values
seed_sequence = "ROMEO: "
temperatures = [0.2, 0.7, 1.2]
models = [
    ("Simple RNN", gen_rnn),
    ("Bidirectional RNN", gen_birnn),
    ("LSTM", gen_lstm),
    ("GRU", gen_gru)
]

print("Generating text across 3 temperatures for all 4 models...\n")

# Run generation loop
generation_results = {}
for model_name, generator in models:
    print(f"{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    generation_results[model_name] = {}

    for temp in temperatures:
        output_text = generator.generate_text(seed_string=seed_sequence, num_generate=200, temperature=temp)
        generation_results[model_name][temp] = output_text
        print(f"\n--- Temperature: {temp} ---")
        print(output_text)
        print("-" * 40)

Generating text across 3 temperatures for all 4 models...

Model: Simple RNN

--- Temperature: 0.2 ---
ROMEO: I have heart a word with the sent to the grown to the dear the father be a death and the bear the common the bear the seater the stand and the father the with a man the come to the counter the great m
----------------------------------------

--- Temperature: 0.7 ---
ROMEO: Lerase madies, you and come and with me, and hold dain some and hearture this Frised
For the wartage planded of trumberse his bedomb a streatur, should stand of hen
brand it well,
Of resent then the c
----------------------------------------

--- Temperature: 1.2 ---
ROMEO: but'ritone frion cuming of the seps.

BUCKItHARTEG EDWARD:
Haight repoor of the brat, she dubsect fordea, lanuer: I lord pursoneing Ableamedfen,
Murds he or, legs their sadvers:
Of send is fome but, b
----------------------------------------
Model: Bidirectional RNN

--- Temperature: 0.2 ---
ROMEO: the the the the the the the the the the 